# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [5]:
import pandas as pd
import numpy as np

import cleaning

In [6]:
customers_df = pd.read_csv('marketing_customer_analysis.csv')
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     10910 non-null  int64  
 1   Customer                       10910 non-null  str    
 2   State                          10279 non-null  str    
 3   Customer Lifetime Value        10910 non-null  float64
 4   Response                       10279 non-null  str    
 5   Coverage                       10910 non-null  str    
 6   Education                      10910 non-null  str    
 7   Effective To Date              10910 non-null  str    
 8   EmploymentStatus               10910 non-null  str    
 9   Gender                         10910 non-null  str    
 10  Income                         10910 non-null  int64  
 11  Location Code                  10910 non-null  str    
 12  Marital Status                 10910 non-null  str    
 1

In [7]:
# standardize column names
customers_df.columns = (
    customers_df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# drop any unnamed exported index columns
customers_df = customers_df.loc[:, ~customers_df.columns.str.startswith("unnamed")]

# strip spaces from text columns
text_cols = customers_df.select_dtypes(include="object").columns
customers_df[text_cols] = customers_df[text_cols].apply(lambda s: s.str.strip())

# convert date column
if "effective_to_date" in customers_df.columns:
    customers_df["effective_to_date"] = pd.to_datetime(
        customers_df["effective_to_date"],
        errors="coerce"
    )

# convert numeric columns
numeric_cols = [
    "customer_lifetime_value",
    "income",
    "monthly_premium_auto",
    "months_since_last_claim",
    "months_since_policy_inception",
    "number_of_open_complaints",
    "number_of_policies",
    "total_claim_amount"
]

existing_numeric_cols = [col for col in numeric_cols if col in customers_df.columns]
customers_df[existing_numeric_cols] = customers_df[existing_numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

# remove duplicates
customers_df = customers_df.drop_duplicates()

# inspect missing values and structure
print(customers_df.isna().sum().sort_values(ascending=False))
print(customers_df.info())
print(customers_df.columns.tolist())


vehicle_type                     5465
number_of_open_complaints         623
months_since_last_claim           623
response                          614
state                             614
vehicle_size                      608
vehicle_class                     608
months_since_policy_inception       0
total_claim_amount                  0
sales_channel                       0
renew_offer_type                    0
policy                              0
policy_type                         0
number_of_policies                  0
customer                            0
marital_status                      0
location_code                       0
income                              0
gender                              0
employmentstatus                    0
effective_to_date                   0
education                           0
coverage                            0
customer_lifetime_value             0
monthly_premium_auto                0
dtype: int64
<class 'pandas.DataFrame'>
Index: 108

/var/folders/b4/svxttqvs5zg_rg_drrxn_vf00000gn/T/ipykernel_16974/3128501173.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = customers_df.select_dtypes(include="object").columns
/var/folders/b4/svxttqvs5zg_rg_drrxn_vf00000gn/T/ipykernel_16974/3128501173.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  customers_df["effective_to_date"] = pd.to_datetime(


In [8]:
customers_df = customers_df.drop_duplicates()

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [9]:
filtered_df = customers_df[
    (customers_df["total_claim_amount"] < 1000) &
    (customers_df["response"] == "Yes")
]
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 1397 entries, 3 to 10897
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   customer                       1397 non-null   str           
 1   state                          1397 non-null   str           
 2   customer_lifetime_value        1397 non-null   float64       
 3   response                       1397 non-null   str           
 4   coverage                       1397 non-null   str           
 5   education                      1397 non-null   str           
 6   effective_to_date              1397 non-null   datetime64[us]
 7   employmentstatus               1397 non-null   str           
 8   gender                         1397 non-null   str           
 9   income                         1397 non-null   int64         
 10  location_code                  1397 non-null   str           
 11  marital_status                 1

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [10]:
filtered_df = customers_df[customers_df["response"].astype(str).str.strip().str.lower() == "yes"]

new_df = filtered_df.groupby(["policy_type", "gender"]).agg(
    avg_monthly_premium=("monthly_premium_auto", "mean"),
    avg_customer_lifetime_value=("customer_lifetime_value", "mean"),
    avg_total_claim_amount=("total_claim_amount", "mean"),
    customers=("customer", "size")
).reset_index()

new_df=new_df.round(2)
new_df

,policy_type,gender,avg_monthly_premium,avg_customer_lifetime_value,avg_total_claim_amount,customers
0,Corporate Auto,F,94.30,7712.63,433.74,169
1,Corporate Auto,M,92.13,7928.36,407.83,153
2,Personal Auto,F,98.96,8339.10,452.76,539
3,Personal Auto,M,91.09,7448.38,457.01,536
4,Special Auto,F,92.31,7691.58,453.28,35
5,Special Auto,M,86.34,8247.09,429.53,32


If your goal is profitability, the strongest segment looks like  Personal Auto  females because they combine the highest premium with the highest CLV.  If your goal is risk control,  Corporate Auto  males stand out because their average total claim amount is the lowest

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [11]:
customers_per_state = (
    customers_df.groupby("state")
    .agg(customers=("customer", "size"))
    .reset_index()
    .sort_values(by="customers", ascending=False)
)

filtered_states = customers_per_state[customers_per_state["customers"] > 500]
filtered_states


,state,customers
1,California,3548
3,Oregon,2897
0,Arizona,1934
2,Nevada,992
4,Washington,888


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [12]:
lifetime_edu_and_gender = customers_df.groupby(["education", "gender"]).agg(
    max_customer_lifetime_value=("customer_lifetime_value", "max"),
    min_customer_lifetime_value=("customer_lifetime_value", "min"),
    median_customer_lifetime_value=("customer_lifetime_value", "median")
    ).reset_index()
lifetime_edu_and_gender=lifetime_edu_and_gender.round(2)
lifetime_edu_and_gender

,education,gender,max_customer_lifetime_value,min_customer_lifetime_value,median_customer_lifetime_value
0,Bachelor,F,73225.96,1904.00,5632.61
1,Bachelor,M,67907.27,1898.01,5548.03
2,College,F,61850.19,1898.68,5623.61
3,College,M,61134.68,1918.12,6005.85
4,Doctor,F,44856.11,2395.57,5332.46
5,Doctor,M,32677.34,2267.60,5577.67
6,High School or Below,F,55277.45,2144.92,6035.09
7,High School or Below,M,83325.38,1940.98,6286.73
8,Master,F,51016.07,2417.78,5729.86
9,Master,M,50568.26,2272.31,5579.10


The median customer lifetime value is relatively similar across education groups, so the typical customer does not differ drastically by education. However, the maximum values vary a lot, which suggests the differences are driven more by outliers than by the overall distribution. In general, education alone does not appear to be a strong predictor of customer lifetime value

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [13]:
# your code goes here